In [1]:
import os
import json
import pandas as pd
import traceback

In [2]:
from langchain.chat_models import ChatOpenAI

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
KEY = os.getenv("OPENAI_API_KEY")

In [5]:
llm = ChatOpenAI(openai_api_key=KEY, model_name="gpt-4o", temperature=0.5)

/var/folders/t9/07qggznn40qf4knqrptd6jc40000gn/T/ipykernel_86301/1616989498.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(openai_api_key=KEY, model_name="gpt-4o", temperature=0.5)


In [7]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [8]:
RESPONSE_JSON = {
    "1" : {
        "MCQ" : "Multiple Choice Question",
        "Options": {
            "A": "Option A",
            "B": "Option B",
            "C": "Option C",
            "D": "Option D"
        },
        "Correct" : "Correct Answer"
    },
    "2" : {
        "MCQ" : "Multiple Choice Question",
        "Options": {
            "A": "Option A",
            "B": "Option B",
            "C": "Option C",
            "D": "Option D"
        },
        "Correct" : "Correct Answer"
    },
    "3" : {
        "MCQ" : "Multiple Choice Question",
        "Options": {
            "A": "Option A",
            "B": "Option B",
            "C": "Option C",
            "D": "Option D"
        },
        "Correct" : "Correct Answer"
    } 
}

In [9]:
TEMPLATE = """
Text: {text}
You are an expert MCQ Generator. Given the above text, your task is to create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure that the questions are relevant to the text and are of varying difficulty levels also take special care that the questions are not repetitive. Check if all the questions conform to the text.
Make sure you format your response like RESPONSE_JSON below and use it like a guide.
Ensure to make {number} questions.
{response_json}
"""

In [10]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text","number","subject","tone","response_json"],
    template = TEMPLATE
)

In [11]:
quiz_chain = LLMChain(llm=llm, prompt = quiz_generation_prompt, output_key = "quiz", verbose = True)

/var/folders/t9/07qggznn40qf4knqrptd6jc40000gn/T/ipykernel_86301/1529459304.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  quiz_chain = LLMChain(llm=llm, prompt = quiz_generation_prompt, output_key = "quiz", verbose = True)


In [12]:
Template2 = """
You are an expert english grammar teacher. Given a multiple choice question quiz for {subject} students. 
You need to evaluate the quiz complexity and provide feedback on the correctness of the questions. Only use 50 words at max for complexity.
If you find any incorrect questions, update the quiz questions which need to be corrected and change the tone so that it perfectly fits 
the student ability.
Ensure to provide feedback for all questions.

Quiz_MCQs: 
{quiz}

Check from an English Writer for the above quiz.
"""

In [13]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject","quiz"],
    template = Template2
)

In [14]:
quiz_evaluation_chain = LLMChain(llm=llm, prompt = quiz_evaluation_prompt, output_key = "review", verbose = True)

In [15]:
generate_eval_chain = SequentialChain(chains=[quiz_chain, quiz_evaluation_chain], input_variables = ["text","number","subject","tone","response_json"],
                                    output_variables = ["quiz","review"], verbose = True)

In [16]:
file_path = r"/Users/atharva7/Downloads/MCQ_Generator/data.txt"

In [17]:
file_path

'/Users/atharva7/Downloads/MCQ_Generator/data.txt'

In [18]:
with open(file_path,'r') as file:
    text = file.read()

In [19]:
print(text)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions.[1] Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.[2]

ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine.[3][4] The application of ML to business problems is known as predictive analytics.

Statistics and mathematical optimization (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.[6][7]

From a theoretical viewpoint, probably approxima

In [20]:
json.dumps(RESPONSE_JSON)

'{"1": {"MCQ": "Multiple Choice Question", "Options": {"A": "Option A", "B": "Option B", "C": "Option C", "D": "Option D"}, "Correct": "Correct Answer"}, "2": {"MCQ": "Multiple Choice Question", "Options": {"A": "Option A", "B": "Option B", "C": "Option C", "D": "Option D"}, "Correct": "Correct Answer"}, "3": {"MCQ": "Multiple Choice Question", "Options": {"A": "Option A", "B": "Option B", "C": "Option C", "D": "Option D"}, "Correct": "Correct Answer"}}'

In [21]:
NUMBER  = 4
SUBJECT = "Machine Learning"
TONE = "Simple"

In [22]:
with get_openai_callback() as cb:
    response = generate_eval_chain(
        {
            "text" : text,
            "number" : NUMBER,
            "subject" : SUBJECT,
            "tone" : TONE,
            "response_json" : json.dumps(RESPONSE_JSON)
        }
    )

/var/folders/t9/07qggznn40qf4knqrptd6jc40000gn/T/ipykernel_86301/317891879.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = generate_eval_chain(




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions.[1] Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.[2]

ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine.[3][4] The application of ML to business problems is known as predictive analytics.

Statistics and mathematical optimization (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on explo

In [23]:
print(f"Total Tokens: {cb.total_tokens}")
print(f"Prompt Tokens: {cb.prompt_tokens}")
print(f"Completion Tokens: {cb.completion_tokens}")
print(f"Total Cost: {cb.total_cost}")

Total Tokens: 3365
Prompt Tokens: 2910
Completion Tokens: 455
Total Cost: 0.011824999999999999


In [24]:
response

{'text': 'Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions.[1] Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.[2]\n\nML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine.[3][4] The application of ML to business problems is known as predictive analytics.\n\nStatistics and mathematical optimization (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.[6][7]\n\nFrom a theoretical viewpoint, pr

In [25]:
response.get("quiz")

'{\n  "1": {\n    "MCQ": "What is machine learning primarily concerned with?",\n    "Options": {\n      "A": "Developing neural networks only",\n      "B": "Learning from data and generalizing to unseen data",\n      "C": "Creating explicit instructions for tasks",\n      "D": "Building physical robots"\n    },\n    "Correct": "B"\n  },\n  "2": {\n    "MCQ": "Which field is closely related to machine learning and focuses on exploratory data analysis?",\n    "Options": {\n      "A": "Data mining",\n      "B": "Artificial intelligence",\n      "C": "Computer vision",\n      "D": "Predictive analytics"\n    },\n    "Correct": "A"\n  },\n  "3": {\n    "MCQ": "Who coined the term \'machine learning\'?",\n    "Options": {\n      "A": "Alan Turing",\n      "B": "Arthur Samuel",\n      "C": "Donald Hebb",\n      "D": "Geoffrey Hinton"\n    },\n    "Correct": "B"\n  },\n  "4": {\n    "MCQ": "What is the primary objective of modern-day machine learning?",\n    "Options": {\n      "A": "To replic

In [27]:
print(type(quiz))  # Should be <class 'str'>

<class 'str'>


In [26]:
quiz = response.get("quiz")

In [30]:
quiz = json.loads(quiz)

In [31]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["MCQ"]
    options = " | ".join(
        [
            f"{option} : {option_value}"
            for option, option_value in value["Options"].items()
        ]
    )
    correct = value["Correct"]
    quiz_table_data.append({"MCQ" : mcq, "Choices" : options, "Correct" : correct})

In [34]:
quiz = pd.DataFrame(quiz_table_data)

In [36]:
quiz

,MCQ,Choices,Correct
0,What is machine learning primarily concerned w...,A : Developing neural networks only | B : Lear...,B
1,Which field is closely related to machine lear...,A : Data mining | B : Artificial intelligence ...,A
2,Who coined the term 'machine learning'?,A : Alan Turing | B : Arthur Samuel | C : Dona...,B
3,What is the primary objective of modern-day ma...,A : To replicate human intelligence | B : To c...,B


In [35]:
quiz.to_csv("ML_Quiz.csv", index = False)